# AI Powered Investment Estimation Agent

In [194]:
import os
from openai import OpenAI
import gradio as gr
import datetime
from dotenv import load_dotenv
import requests
import json
import google.genai as genai
from google.genai import types
import wave

In [254]:
load_dotenv()

gemini_api_key = os.getenv("GOOGLE_API_KEY")

if gemini_api_key:
    print(f"Gemini API key has been found with the initials as {gemini_api_key[:8]}")
else:
    print(f"No api key has been found!")
    
MODEL = "gemini-2.5-flash"
gemini_via_openai = OpenAI(
    api_key=gemini_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/models/"
)

Gemini API key has been found with the initials as AIzaSyC5


In [196]:
gold_api_key = os.getenv("GOLD_API_KEY", "demo")
GOLD_API_URL = "https://api.metalpriceapi.com/v1/latest"

if gold_api_key:
    print(f"Gold API key has been found with the initials as {gold_api_key[:8]}")
else:
    print("No Gold API key has been found!")

Gold API key has been found with the initials as 29fc54e8


In [197]:
system_prompt = "You are a helpful assistant to an AI powered gold investment estimation company specializing in Gold investments."
system_prompt += "Give an informative, courteous answers about gold prices and investment advice."
system_prompt += "Always be accurate, if you don't know the answer, say so"
system_prompt += "Provide investment recommendation based on the current market conditions."

In [198]:
# Gold Price Functions
def get_gold_price(country="USD"):
    """Get a gold price based on the specific input country/currency"""
    print("Tool called get_gold_price with country {country}")
    
    country_currency_map = {
        "united states": "USD",
        "eurozone": "EUR",
        "united kingdom": "GBP",
        "india": "INR",
        "japan": "JPY",
        "australia": "AUD",
        "canada": "CAD",
        "switzerland": "CHF",
        "china": "CNY",
        "singapore": "SGD",
        "south africa": "ZAR",
        "russia": "RUB",
        "brazil": "BRL",
        "mexico": "MXN",
        "saudi arabia": "SAR",
        "turkey": "TRY",
        "south korea": "KRW",
        "indonesia": "IDR",
        "thailand": "THB",
        "new zealand": "NZD",
        "sweden": "SEK",
        "norway": "NOK",
        "denmark": "DKK",
        "poland": "PLN",
        "hong kong": "HKD",
        "malaysia": "MYR",
        "united arab emirates": "AED",
        "pakistan": "PKR",
        "egypt": "EGP"
    }
    
    currency = country_currency_map.get(country.lower(), "USD")
    
    # Demo prices for currencies
    demo_prices = {
        "USD": 2350.0,
        "EUR": 2150.0,
        "GBP": 2000.0,
        "INR": 195000.0,
        "JPY": 340000.0,
        "AUD": 3550.0,
        "CAD": 3200.0,
        "CHF": 2100.0,
        "CNY": 17000.0,
        "SGD": 3200.0,
        "ZAR": 44000.0,
        "RUB": 210000.0,
        "BRL": 12000.0,
        "MXN": 40000.0,
        "SAR": 8800.0,
        "TRY": 76000.0,
        "KRW": 3200000.0,
        "IDR": 37000000.0,
        "THB": 85000.0,
        "NZD": 3800.0,
        "SEK": 24000.0,
        "NOK": 25000.0,
        "DKK": 16000.0,
        "PLN": 9000.0,
        "HKD": 18400.0,
        "MYR": 11000.0,
        "AED": 8600.0,
        "PKR": 660000.0,
        "EGP": 115000.0
    }
    
    price_per_ounce = None
    api_success = False
    
    try:
        # API call to get gold price
        params = {
            'api_key': gold_api_key,
            'base': 'XAU',  # Gold symbol
            'currencies': currency
        }
        
        response = requests.get(GOLD_API_URL, params=params, timeout=10)
        print(f"API Response Status: {response.status_code}")
        
        if response.status_code == 200:
            data = response.json()
            print(f"API Response Data: {data}")
            
            if 'rates' in data and currency in data['rates']:
                rate = data['rates'][currency]
                if rate > 0:  # Ensure valid rate
                    price_per_ounce = round(rate, 2)  # Rate is already price per ounce
                    api_success = True
                    print(f"Successfully got price from API: {price_per_ounce}")
                else:
                    print(f"Invalid rate from API: {rate}")
            else:
                print(f"Currency {currency} not found in API response")
        else:
            print(f"API request failed with status {response.status_code}: {response.text}")
            
    except Exception as e:
        print(f"Error fetching gold price from API: {e}")
    
    # Use demo data if API failed
    if price_per_ounce is None:
        price_per_ounce = demo_prices.get(currency, 2350.50)
        print(f"Using demo price: {price_per_ounce}")
        
    # Generate AI-powered investment advice with risk level assessment
    advice = generate_ai_investment_advice(price_per_ounce, currency, country)
    
    return {
        'price': price_per_ounce,
        'currency': currency,
        'country': country,
        'advice': advice,
        'timestamp': datetime.datetime.now().isoformat(),
        'data_source': 'API' if api_success else 'Demo'
    }
    
def generate_ai_investment_advice(price, currency, country):
    """Generate AI-powered investment advice using risk level assessment as a tool"""
    try:
        # First, get the risk level assessment from our analysis tool
        risk_analysis = get_risk_level_assessment(price, currency)
        risk_level = risk_analysis['risk_level']
        fallback_advice = risk_analysis['advice']
        
        # Create enhanced context for the AI advice using risk assessment as a tool
        prompt = f"""
        As a professional gold investment advisor, provide specific investment advice based on these current market conditions:
        
        MARKET DATA:
        - Current gold price: {price} {currency}
        - Country/Market: {country}
        - Currency: {currency}
        
        RISK ASSESSMENT TOOL OUTPUT:
        - Risk Level: {risk_level.upper()}
        - Price Classification: {risk_analysis['price_classification']}
        - Technical Analysis: {fallback_advice}
        
        Based on this risk assessment tool output, provide professional investment advice that:
        1. Acknowledges the {risk_level} risk level
        2. Provides specific timing recommendations
        3. Considers currency-specific factors for {currency}
        4. Offers actionable next steps
        
        Keep your advice concise (2-3 sentences), professional, and specific to the {risk_level} risk scenario.
        """
        
        response = gemini_via_openai.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You are a professional gold investment advisor. Use the risk assessment tool output to provide specific, actionable advice. Build upon the technical analysis provided."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=200,
            temperature=0.7
        )
        
        return response.choices[0].message.content.strip()
        
    except Exception as e:
        print(f"Error generating AI advice: {e}")
        # Fallback to rule-based advice if AI fails
        return generate_fallback_advice(price, currency)
        
def get_risk_level_assessment(price, currency):
    """Risk assessment tool that determines price level and provides technical analysis"""
    
    # Currency-specific price thresholds (approximate)
    thresholds = {
        'USD': {'low': 2000, 'moderate': 2300, 'high': 2500},
        'EUR': {'low': 1850, 'moderate': 2150, 'high': 2350},
        'GBP': {'low': 1600, 'moderate': 1850, 'high': 2100},
        'INR': {'low': 160000, 'moderate': 190000, 'high': 220000},
        'JPY': {'low': 300000, 'moderate': 340000, 'high': 380000},
        'AUD': {'low': 3000, 'moderate': 3500, 'high': 4000},
        'CAD': {'low': 2700, 'moderate': 3100, 'high': 3500},
        'CHF': {'low': 1800, 'moderate': 2100, 'high': 2400},
        'CNY': {'low': 14000, 'moderate': 16500, 'high': 19000},
        'SGD': {'low': 2700, 'moderate': 3200, 'high': 3700},
        'ZAR': {'low': 35000, 'moderate': 42000, 'high': 48000},
        'RUB': {'low': 170000, 'moderate': 200000, 'high': 230000},
        'BRL': {'low': 9000, 'moderate': 11000, 'high': 13000},
        'MXN': {'low': 32000, 'moderate': 38000, 'high': 44000},
        'SAR': {'low': 7500, 'moderate': 8500, 'high': 9500},
        'TRY': {'low': 60000, 'moderate': 72000, 'high': 84000},
        'KRW': {'low': 2600000, 'moderate': 3100000, 'high': 3600000},
        'IDR': {'low': 30000000, 'moderate': 35000000, 'high': 40000000},
        'THB': {'low': 70000, 'moderate': 80000, 'high': 90000},
        'NZD': {'low': 3200, 'moderate': 3700, 'high': 4200},
        'SEK': {'low': 20000, 'moderate': 23000, 'high': 26000},
        'NOK': {'low': 21000, 'moderate': 24000, 'high': 27000},
        'DKK': {'low': 13000, 'moderate': 15000, 'high': 17000},
        'PLN': {'low': 7000, 'moderate': 8500, 'high': 10000},
        'HKD': {'low': 15000, 'moderate': 17000, 'high': 19000},
        'MYR': {'low': 9000, 'moderate': 10500, 'high': 12000},
        'AED': {'low': 7300, 'moderate': 8300, 'high': 9300},
        'PKR': {'low': 500000, 'moderate': 600000, 'high': 700000},
        'EGP': {'low': 95000, 'moderate': 110000, 'high': 125000}
    }
    
    # Get thresholds for currency or use USD as default
    thresh = thresholds.get(currency, thresholds['USD'])
    
    if price < thresh['low']:
        return {
            'risk_level': 'low',
            'price_classification': 'Undervalued',
            'advice': f"Excellent buying opportunity! Gold is undervalued at {price} {currency}. Consider accumulating positions while prices are low."
        }
    elif price < thresh['moderate']:
        return {
            'risk_level': 'moderate', 
            'price_classification': 'Fair Value',
            'advice': f"Good entry point at {price} {currency}. Moderate pricing with growth potential. Consider dollar-cost averaging for this market."
        }
    elif price < thresh['high']:
        return {
            'risk_level': 'moderate-high',
            'price_classification': 'Fairly Valued',
            'advice': f"Fair pricing at {price} {currency}. Market is fairly valued. Consider smaller purchases or wait for pullbacks."
        }
    else:
        return {
            'risk_level': 'high',
            'price_classification': 'Premium/Overvalued',
            'advice': f"Premium pricing at {price} {currency}. Consider waiting for market corrections or focus on smaller strategic purchases."
        }

def generate_fallback_advice(price, currency):
    """Enhanced fallback advice with currency-specific considerations"""
    
    # Currency-specific price thresholds (approximate)
    thresholds = {
        'USD': {'low': 2000, 'moderate': 2300, 'high': 2500},
        'EUR': {'low': 1850, 'moderate': 2150, 'high': 2350},
        'GBP': {'low': 1600, 'moderate': 1850, 'high': 2100},
        'INR': {'low': 160000, 'moderate': 190000, 'high': 220000},
        'JPY': {'low': 300000, 'moderate': 340000, 'high': 380000},
        'AUD': {'low': 3000, 'moderate': 3500, 'high': 4000},
        'CAD': {'low': 2700, 'moderate': 3100, 'high': 3500},
        'CHF': {'low': 1800, 'moderate': 2100, 'high': 2400},
        'CNY': {'low': 14000, 'moderate': 16500, 'high': 19000},
        'SGD': {'low': 2700, 'moderate': 3200, 'high': 3700},
        'ZAR': {'low': 35000, 'moderate': 42000, 'high': 48000},
        'RUB': {'low': 170000, 'moderate': 200000, 'high': 230000},
        'BRL': {'low': 9000, 'moderate': 11000, 'high': 13000},
        'MXN': {'low': 32000, 'moderate': 38000, 'high': 44000},
        'SAR': {'low': 7500, 'moderate': 8500, 'high': 9500},
        'TRY': {'low': 60000, 'moderate': 72000, 'high': 84000},
        'KRW': {'low': 2600000, 'moderate': 3100000, 'high': 3600000},
        'IDR': {'low': 30000000, 'moderate': 35000000, 'high': 40000000},
        'THB': {'low': 70000, 'moderate': 80000, 'high': 90000},
        'NZD': {'low': 3200, 'moderate': 3700, 'high': 4200},
        'SEK': {'low': 20000, 'moderate': 23000, 'high': 26000},
        'NOK': {'low': 21000, 'moderate': 24000, 'high': 27000},
        'DKK': {'low': 13000, 'moderate': 15000, 'high': 17000},
        'PLN': {'low': 7000, 'moderate': 8500, 'high': 10000},
        'HKD': {'low': 15000, 'moderate': 17000, 'high': 19000},
        'MYR': {'low': 9000, 'moderate': 10500, 'high': 12000},
        'AED': {'low': 7300, 'moderate': 8300, 'high': 9300},
        'PKR': {'low': 500000, 'moderate': 600000, 'high': 700000},
        'EGP': {'low': 95000, 'moderate': 110000, 'high': 125000}
    }
    
    # Get thresholds for currency or use USD as default
    thresh = thresholds.get(currency, thresholds['USD'])
    
    if price < thresh['low']:
        return f"Excellent buying opportunity! Gold is undervalued at {price} {currency}. Consider accumulating positions while prices are low."
    elif price < thresh['moderate']:
        return f"Good entry point at {price} {currency}. Moderate pricing with growth potential. Consider dollar-cost averaging for this market."
    elif price < thresh['high']:
        return f"Fair pricing at {price} {currency}. Market is fairly valued. Consider smaller purchases or wait for pullbacks."
    else:
        return f"Premium pricing at {price} {currency}. Consider waiting for market corrections or focus on smaller strategic purchases."

In [199]:
print("Testing AI-Enhanced Gold Price Function with Risk Assessment Tool:")
print("=" * 70)

# Test USA (likely high risk due to current high prices)
test_result = get_gold_price("USA")
print(f"\n🇺🇸 USA Test:")
print(f"Price: {test_result['price']} {test_result['currency']}")
print(f"Data Source: {test_result.get('data_source', 'Unknown')}")
print(f"AI Advice: {test_result['advice']}")

# Test a few more currencies to show different risk levels
print(f"\n{'='*50}")
print("Testing Different Currencies with AI Risk Assessment:")

test_currencies = ["USA", "UK", "EUR", "JPY"]
for currency in test_currencies:
    result = get_gold_price(currency)
    
    # Also show the risk assessment tool output
    risk_assessment = get_risk_level_assessment(result['price'], result['currency'])
    
    print(f"\n🌍 {currency}:")
    print(f"  Price: {result['price']} {result['currency']}")
    print(f"  Risk Level: {risk_assessment['risk_level'].upper()}")
    print(f"  Classification: {risk_assessment['price_classification']}")
    print(f"  AI Advice: {result['advice']}")
    print(f"  Data Source: {result.get('data_source', 'Unknown')}")


Testing AI-Enhanced Gold Price Function with Risk Assessment Tool:
Tool called get_gold_price with country {country}


API Response Status: 200
API Response Data: {'success': True, 'base': 'XAU', 'timestamp': 1758067199, 'rates': {'USD': 3678.4626380389, 'XAUUSD': 0.0002718527}}
Successfully got price from API: 3678.46
Error generating AI advice: Error code: 404

🇺🇸 USA Test:
Price: 3678.46 USD
Data Source: API
AI Advice: Premium pricing at 3678.46 USD. Consider waiting for market corrections or focus on smaller strategic purchases.

Testing Different Currencies with AI Risk Assessment:
Tool called get_gold_price with country {country}
Error generating AI advice: Error code: 404

🇺🇸 USA Test:
Price: 3678.46 USD
Data Source: API
AI Advice: Premium pricing at 3678.46 USD. Consider waiting for market corrections or focus on smaller strategic purchases.

Testing Different Currencies with AI Risk Assessment:
Tool called get_gold_price with country {country}
API Response Status: 200
API Response Data: {'success': True, 'base': 'XAU', 'timestamp': 1758067199, 'rates': {'USD': 3678.4626380389, 'XAUUSD': 0.0002

In [200]:
# Fake Gold Buying Simulation
def simulate_gold_purchase(ounces, country="USD"):
    """Simulate buying gold and write to file"""
    print(f"Tool simulate_gold_purchase called for {ounces} ounces in {country}")
    
    try:
        # Get current price
        price_data = get_gold_price(country)
        
        if 'error' in price_data:
            return {'error': price_data['error']}
        
        # Ensure we have valid price data
        if 'price' not in price_data or price_data['price'] <= 0:
            return {'error': 'Invalid price data received'}
        
        price_per_ounce = float(price_data['price'])
        currency = price_data['currency']
        ounces_float = float(ounces)
        
        # Calculate total cost
        total_cost = round(price_per_ounce * ounces_float, 2)
        
        # Create detailed purchase record
        purchase_record = {
            'purchase_date': datetime.datetime.now().isoformat(),
            'ounces': ounces_float,
            'price_per_ounce': price_per_ounce,
            'total_cost': total_cost,
            'currency': currency,
            'country': country,
            'transaction_id': f"GOLD_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}",
            'price_source': price_data.get('source', 'Unknown'),
            'market_advice': price_data.get('advice', 'No advice available')
        }
        
        # Write to file
        filename = "gold_purchases.json"
        
        # Load existing purchases or create new list
        try:
            with open(filename, 'r') as f:
                purchases = json.load(f)
        except FileNotFoundError:
            purchases = []
        
        # Add new purchase
        purchases.append(purchase_record)
        
        # Save back to file
        with open(filename, 'w') as f:
            json.dump(purchases, f, indent=2)
        
        # Create detailed success message
        success_message = f"✅ Successfully purchased {ounces} ounces of gold for {total_cost:,.2f} {currency}"
        if 'note' in price_data:
            success_message += f" ({price_data['note']})"
        
        return {
            'success': True,
            'message': success_message,
            'transaction_id': purchase_record['transaction_id'],
            'total_cost': total_cost,
            'currency': currency,
            'price_per_ounce': price_per_ounce,
            'market_advice': price_data.get('advice', 'No advice available')
        }
        
    except ValueError as e:
        return {'error': f'Invalid number format: {str(e)}'}
    except Exception as e:
        return {'error': f'Purchase failed: {str(e)}'}

In [201]:
# Test the purchase simulation
print("Testing purchase simulation...")
test_purchase = simulate_gold_purchase("2.5", "united states")
print("Purchase Result:")
print(json.dumps(test_purchase, indent=2))

Testing purchase simulation...
Tool simulate_gold_purchase called for 2.5 ounces in united states
Tool called get_gold_price with country {country}
API Response Status: 200
API Response Data: {'success': True, 'base': 'XAU', 'timestamp': 1758067199, 'rates': {'USD': 3678.4626380389, 'XAUUSD': 0.0002718527}}
Successfully got price from API: 3678.46
Error generating AI advice: Error code: 404
Purchase Result:
{
  "success": true,
  "message": "\u2705 Successfully purchased 2.5 ounces of gold for 9,196.15 USD",
  "transaction_id": "GOLD_20250917_142955",
  "total_cost": 9196.15,
  "currency": "USD",
  "price_per_ounce": 3678.46,
  "market_advice": "Premium pricing at 3678.46 USD. Consider waiting for market corrections or focus on smaller strategic purchases."
}
API Response Status: 200
API Response Data: {'success': True, 'base': 'XAU', 'timestamp': 1758067199, 'rates': {'USD': 3678.4626380389, 'XAUUSD': 0.0002718527}}
Successfully got price from API: 3678.46
Error generating AI advice: 

In [202]:
# Translation Agent
def translate_to_arabic(text):
    """Translate text to Arabic using Gemini"""
    
    print(f"translate to arabic: {text}")
    
    try:
        # Use native Google Generative AI client instead of OpenAI interface
        client = genai.Client()
        
        response = client.models.generate_content(
            model="gemini-2.0-flash-exp",
            contents=f"Translate the following text to Arabic. Maintain the meaning and tone. Only return the Arabic translation, nothing else:\n\n{text}"
        )
        
        translated_text = response.candidates[0].content.parts[0].text
        print(f"Result of translate to arabic: {translated_text}")
        return translated_text
        
    except Exception as e:
        print(f"Translation error: {str(e)}")
        # Return original text if translation fails, so TTS can still work
        print("Falling back to original English text for TTS")
        return text

In [203]:
def wave_file(filename, pcm, channels=1, rate=24000, sample_width=2):
    """Helper function to save audio data as WAV file"""
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        wf.writeframes(pcm)
    
# Text-to-Speech Agent
def text_to_speech(text, filename="speech_output.wav"):
    """Convert text to speech using Google Generative AI Gemini TTS"""
    
    print(f"text to speech text: {text}")
    
    try:
        client = genai.Client()
        
        response = client.models.generate_content(
        model="gemini-2.5-flash-preview-tts",
        contents=text,
        config=types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=types.SpeechConfig(
                voice_config=types.VoiceConfig(
                    prebuilt_voice_config=types.PrebuiltVoiceConfig(
                    voice_name='Kore',
                    )
                )
            ),
        )
        )

        data = response.candidates[0].content.parts[0].inline_data.data

        wave_file(filename, data) # Saves the file to current directory
        
        return {
            'success': True,
            'filename': filename,
            'audio_data': data,
            'message': f'Speech generated successfully and saved as {filename}'
        }
        
    except Exception as e:
        return {
            'success': False,
            'error': f"Text-to-speech error: {str(e)}",
            'message': 'Failed to generate speech'
        }

In [204]:
from IPython.display import Audio
import os

def play_tts(text_content):
    print(f"Trying to convert to speech: {text_content}")

    # Try to convert to speech using the new Google Generative AI function
    tts_result = text_to_speech(text_content, f"{text_content[:12].lower()}.wav")

    if tts_result['success']:
        print(f"✅ {tts_result['message']}")
        if os.path.exists(tts_result['filename']):
            print(f"📁 Audio file saved: {tts_result['filename']}")
            try:
                display(Audio(tts_result['filename']))
                print("🔊 Audio player displayed above")
            except Exception as e:
                print(f"⚠️ Could not display audio player: {e}")
                print("📝 You can manually play the audio file from the file system")
        else:
            print("❌ Audio file was not created")
    else:
        print(f"❌ Text-to-speech failed: {tts_result.get('error', 'Unknown error')}")
        print("💡 This might be due to:")
        print("   - Missing google-generativeai library (try: pip install google-generativeai)")
        print("   - API key issues")
        print("   - Network connectivity problems")

In [205]:
play_tts(test_purchase["market_advice"])

Trying to convert to speech: Premium pricing at 3678.46 USD. Consider waiting for market corrections or focus on smaller strategic purchases.
text to speech text: Premium pricing at 3678.46 USD. Consider waiting for market corrections or focus on smaller strategic purchases.
✅ Speech generated successfully and saved as premium pric.wav
📁 Audio file saved: premium pric.wav
✅ Speech generated successfully and saved as premium pric.wav
📁 Audio file saved: premium pric.wav


🔊 Audio player displayed above


In [206]:
text_in_arabic = translate_to_arabic(test_purchase["market_advice"])
play_tts(text_in_arabic)

translate to arabic: Premium pricing at 3678.46 USD. Consider waiting for market corrections or focus on smaller strategic purchases.
Result of translate to arabic: تسعير ممتاز بسعر 3678.46 دولارًا أمريكيًا. يُنصح بالانتظار لتصحيحات السوق أو التركيز على عمليات شراء استراتيجية أصغر.

Trying to convert to speech: تسعير ممتاز بسعر 3678.46 دولارًا أمريكيًا. يُنصح بالانتظار لتصحيحات السوق أو التركيز على عمليات شراء استراتيجية أصغر.

text to speech text: تسعير ممتاز بسعر 3678.46 دولارًا أمريكيًا. يُنصح بالانتظار لتصحيحات السوق أو التركيز على عمليات شراء استراتيجية أصغر.

Result of translate to arabic: تسعير ممتاز بسعر 3678.46 دولارًا أمريكيًا. يُنصح بالانتظار لتصحيحات السوق أو التركيز على عمليات شراء استراتيجية أصغر.

Trying to convert to speech: تسعير ممتاز بسعر 3678.46 دولارًا أمريكيًا. يُنصح بالانتظار لتصحيحات السوق أو التركيز على عمليات شراء استراتيجية أصغر.

text to speech text: تسعير ممتاز بسعر 3678.46 دولارًا أمريكيًا. يُنصح بالانتظار لتصحيحات السوق أو التركيز على عمليات شراء استراتيج

🔊 Audio player displayed above


In [208]:
# Define Tools for OpenAI
gold_price_function = {
    "name": "get_gold_price",
    "description": "Get the current price of gold for a specific country/currency and investment advice. Call this when user asks about gold prices or investment timing.",
    "parameters": {
        "type": "object",
        "properties": {
            "country": {
                "type": "string",
                "description": "The country or currency code (e.g., 'USA', 'UK', 'EUR', 'JPY')",
            },
        },
        "required": ["country"],
        "additionalProperties": False
    }
}

gold_purchase_function = {
    "name": "simulate_gold_purchase",
    "description": "Simulate purchasing gold and record the transaction. Call this when user wants to buy gold.",
    "parameters": {
        "type": "object",
        "properties": {
            "ounces": {
                "type": "string",
                "description": "Number of ounces of gold to purchase",
            },
            "country": {
                "type": "string",
                "description": "The country or currency for the purchase (e.g., 'USA', 'UK')",
            },
        },
        "required": ["ounces"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": gold_price_function},
    {"type": "function", "function": gold_purchase_function}
]


In [214]:
# Handle Tool Calls
def handle_tool_call(function_name, arguments):
    """Handle tool calls from Gemini"""
    
    if function_name == "get_gold_price":
        country = arguments.get('country', 'USD')
        result = get_gold_price(country)
    elif function_name == "simulate_gold_purchase":
        ounces = arguments.get('ounces')
        country = arguments.get('country', 'USD')
        result = simulate_gold_purchase(ounces, country)
    else:
        result = {"error": "Unknown function"}
    
    return result

In [215]:
# Main chat function
def chat(history):
    """Main chat function with multi-agent capabilities"""
    messages = [{"role": "system", "content": system_prompt}] + history
    response = gemini_via_openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_response, tool_result = handle_tool_call(message)
        messages.append(message)
        messages.append(tool_response)
        response = gemini_via_openai.chat.completions.create(model=MODEL, messages=messages)
        
    reply = response.choices[0].message.content
    history += [{"role": "assistant", "content": reply}]
    
    arabic_translation = translate_to_arabic(reply)
    
    return history, arabic_translation

In [216]:
# Initialize chat history
chat_history = []

def handle_text_input(message, history):
    if not message.strip():
        return history, "", ""
    history += [{"role": "user", "content": message}]
    updated_history, arabic_translation = chat(history)
    return updated_history, "", arabic_translation

def clear_chat():
    return [], "", ""

with gr.Blocks(title="AI Investment Estimations - Gold Assistant", theme=gr.themes.Soft()) as ui:
    gr.Markdown("# 🏆 AI Investment Estimations - Gold Assistant")
    gr.Markdown("Get real-time gold prices, investment advice, simulate purchases, and interact in multiple languages!")
    
    with gr.Row():
        # Left Panel - English Chat
        with gr.Column(scale=2):
            gr.Markdown("### 💬 English Chat")
            chatbot = gr.Chatbot(height=400, type="messages", label="Investment Assistant")
            
            with gr.Row():
                text_input = gr.Textbox(
                    label="Ask about gold prices, investment advice, or make a purchase",
                    placeholder="e.g., What's the gold price in USA? Should I invest now? Buy 2 ounces of gold"
                )
            
            with gr.Row():
                send_btn = gr.Button("Send Message", variant="primary")
                clear_btn = gr.Button("Clear Chat")
        
        # Right Panel - Arabic Translation
        with gr.Column(scale=1):
            gr.Markdown("### 🌐 Arabic Translation")
            gr.Markdown("الترجمة العربية")
            arabic_output = gr.Textbox(
                label="Arabic Response",
                lines=15,
                interactive=False,
                text_align="right",
                rtl=True
            )
    
    # Bottom Panel - Purchase History
    with gr.Row():
        gr.Markdown("### 📊 Recent Features")
        
    with gr.Row():
        with gr.Column():
            gr.Markdown("**Supported Countries/Currencies:**")
            gr.Markdown("USA (USD), UK (GBP), Europe (EUR), Japan (JPY), Canada (CAD), Australia (AUD), India (INR), China (CNY), Saudi Arabia (SAR), UAE (AED), Egypt (EGP)")
        
        with gr.Column():
            gr.Markdown("**Features:**")
            gr.Markdown("• Real-time gold prices\n• Investment timing advice\n• Simulated gold purchasing\n• Arabic translation")
    
    # Connect events
    text_input.submit(
        handle_text_input,
        inputs=[text_input, chatbot],
        outputs=[chatbot, text_input, arabic_output]
    )
    
    send_btn.click(
        handle_text_input,
        inputs=[text_input, chatbot],
        outputs=[chatbot, text_input, arabic_output]
    )
    
    clear_btn.click(
        clear_chat,
        outputs=[chatbot, text_input, arabic_output]
    )

ui.launch(inbrowser=True, share=False)


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


translate to arabic: Hello! How can I help you with your gold investment questions today? I can provide information on current gold prices, historical trends, different ways to invest in gold (like physical gold, gold ETFs, or gold mining stocks), and discuss how gold can fit into your overall investment portfolio. I can also provide investment recommendations based on the current market conditions. Just let me know what you're interested in!

Result of translate to arabic: مرحباً! كيف يمكنني مساعدتك فيما يتعلق بأسئلتك حول الاستثمار في الذهب اليوم؟ يمكنني تزويدك بمعلومات حول أسعار الذهب الحالية، والاتجاهات التاريخية، والطرق المختلفة للاستثمار في الذهب (مثل الذهب المادي، وصناديق الاستثمار المتداولة في الذهب، أو أسهم شركات تعدين الذهب)، ومناقشة كيف يمكن للذهب أن يتناسب مع محفظتك الاستثمارية بشكل عام. يمكنني أيضاً تقديم توصيات استثمارية بناءً على ظروف السوق الحالية. فقط أخبرني بما تهتم به!

Result of translate to arabic: مرحباً! كيف يمكنني مساعدتك فيما يتعلق بأسئلتك حول الاستثمار في الذهب